In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")


In [2]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001386A8CB080>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013869F52E70>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

result=model.invoke(messages)
result

AIMessage(content='Bonjour, comment ça va?', additional_kwargs={'reasoning_content': 'We need to translate "Hello How are you?" to French. The user says: "Hello How are you?" It\'s a greeting. In French, that would be: "Bonjour, comment ça va ?" Or "Bonjour, comment vas-tu?" The user didn\'t specify formality. The typical translation: "Bonjour, comment ça va ?" That\'s the simplest. Probably "Bonjour, comment vas-tu?" If formal: "Bonjour, comment allez-vous?" The user didn\'t specify. The user wrote "Hello How are you?" in English. The translation: "Bonjour, comment ça va?" So answer.'}, response_metadata={'token_usage': {'completion_tokens': 135, 'prompt_tokens': 86, 'total_tokens': 221, 'completion_time': 0.161073877, 'completion_tokens_details': {'reasoning_tokens': 120}, 'prompt_time': 0.004694973, 'prompt_tokens_details': None, 'queue_time': 0.059145637, 'total_time': 0.16576885}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand

In [4]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Bonjour, comment ça va?'

In [6]:
### Using LCEL- chain the components
chain=model|parser
chain.invoke(messages)

'Bonjour, comment allez‑vous?'

In [10]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Translate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)



In [11]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [12]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [15]:
##Chaining together components with LCEL
chain=prompt|model|parser
chain.invoke({"language":"French","text":"My name is Shrey?"})

"Je m'appelle Shrey?"